# 08: synteza obrazu wprost: zwykły U-Net od zera (długo), autoenkoder, enkoder + korelacja

Trzy modele **bez RAFT i bez geometrii**, na tym samym zbiorze `rs_video_x4k` co 04–07 (porównywalne liczby):

| sekcja | model | `--mode` | wagi | co sprawdza |
|---|---|---|---|---|
| 3 | zwykły U-Net (z 06), **200 epok** zamiast 30 | `plain` | 2,7 M | czy 28,45 dB to tylko za krótki trening |
| 4 | klasyczny autoenkoder **bez skipów**, latent 16×16×512 | `ae` | 10,2 M | co da samo wąskie gardło (spodziewany rozmyty obraz) |
| 5 | syjamski enkoder + **warstwa korelacji** A↔B (±32 px na 1/8, ±12 px na 1/4) + dekoder ze skipami | `corr` | 6,2 M | czy jawne porównanie A z B wystarczy bez geometrii |

Wszystkie dostają RS A, RS B (obrócony z powrotem), współrzędne i czasy odczytu każdego piksela w A i B.
`plain` zwraca pole przesunięć dla A, `ae` i `corr` **zwracają obraz** (synteza wprost: `ae` od zera, `corr` jako A + poprawka).

**Porównuj kolumną `psnr_full` (cały obraz)**: `ae`/`corr` nie mają pola przesunięć, więc ich `psnr` = `psnr_full`.
Punkt odniesienia: RAFT + geometria + U-Net `psnr_full` 35,37 dB, zwykły U-Net (30 epok) 28,41 dB, wejście 26,4 dB.

Harmonogram OneCycle: **liczbę epok ustala się na starcie**, nie da się jej potem wydłużyć (`RESUME` tylko po rozłączeniu).
Sekcje 3–5 są niezależne: można je puścić w osobnych sesjach Colaba naraz (w każdej: SETUP, 1, 2, potem swoja sekcja).
Kod: `src/raft_unet.py`, `src/train_unet.py`. **Przed uruchomieniem zrób commit + push.**

In [ ]:
# === SETUP: uruchamiaj zawsze jako pierwszą komórkę ===
# Kod jest pobierany z GitHuba na maszynę Colaba (lokalne pliki z VS Code tam nie trafiają).
# Po zmianie kodu: commit + push w VS Code, potem ponownie ta komórka (robi git pull).
REPO = 'kordiusz/RollingShutter'   # repozytorium na GitHubie
BRANCH = 'main'

import os, sys, subprocess, getpass, importlib
from google.colab import drive
drive.mount('/content/drive')

TOKEN_FILE = '/content/drive/MyDrive/rs_project/.github_token'   # opcjonalnie: token zapisany na Drive
CODE = '/content/code'

def _git(*args):
    r = subprocess.run(['git', *args], capture_output=True, text=True)
    if r.returncode != 0:
        msg = (r.stderr or r.stdout).replace(token, '***') if 'token' in globals() else r.stderr
        raise RuntimeError('git ' + args[0] + ' nie powiodło się:\n' + msg)
    return r.stdout

if not os.path.exists(CODE):
    token = (open(TOKEN_FILE).read().strip() if os.path.exists(TOKEN_FILE)
             else getpass.getpass('GitHub token (fine-grained, tylko odczyt): '))
    _git('clone', '-q', '-b', BRANCH, f'https://{token}@github.com/{REPO}.git', CODE)
else:
    _git('-C', CODE, 'pull', '-q')
os.chdir(CODE)
sys.path.insert(0, f'{CODE}/src')
import project_paths as P
importlib.reload(P)
P.ensure_dirs()
print(_git('log', '-1', '--format=kod: %h  %s  (%cd)', '--date=short'))
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

## 1. Zbiór danych (rozpakowanie z Drive na dysk maszyny Colaba)

In [ ]:
import os, glob, shutil, subprocess
DATASET_NAME = 'rs_video_x4k'
z = P.find_dataset_zip(DATASET_NAME)
assert z, f'Brak {DATASET_NAME}.zip w {P.DATASETS} - wrzuć tam spakowany folder data/{DATASET_NAME}'
UNZ = f'/content/unz_{DATASET_NAME}'
DONE = f'{UNZ}/.rozpakowane'          # znacznik: rozpakowanie skończyło się bez błędu
if not os.path.exists(DONE):
    shutil.rmtree(UNZ, ignore_errors=True)            # resztki po przerwanym rozpakowaniu
    LOCAL_ZIP = f'/content/{DATASET_NAME}.zip'
    # najpierw kopia na dysk Colaba: unzip czytający wprost z Drive potrafi się urwać ("zipfile read error")
    print(f'kopiuję {z} ({os.path.getsize(z):,} B) na dysk Colaba ...')
    subprocess.run(['cp', str(z), LOCAL_ZIP], check=True)
    t = subprocess.run(['unzip', '-tq', LOCAL_ZIP], capture_output=True, text=True)
    assert t.returncode == 0, ('Archiwum uszkodzone - wgraj zip na Drive ponownie (lokalnie ma 1 233 434 992 B):\n'
                               + (t.stdout + t.stderr)[-1500:])
    print('archiwum OK, rozpakowuję ...')
    subprocess.run(['unzip', '-q', LOCAL_ZIP, '-d', UNZ], check=True)
    os.remove(LOCAL_ZIP)
    open(DONE, 'w').close()
# folder z podfolderami train/ i val/ (niezależnie od tego, jak spakowano)
DATA = next(os.path.dirname(p) for p in sorted(glob.glob(f'{UNZ}/**/train', recursive=True)))
assert os.path.isdir(f'{DATA}/val'), f'brak {DATA}/val - usuń {UNZ} i uruchom komórkę ponownie'
print('DATA =', DATA, '|', len(os.listdir(f'{DATA}/train')), 'train /', len(os.listdir(f'{DATA}/val')), 'val')
WORKERS = min(8, os.cpu_count())

## 2. Nazwy runów
Przy wznawianiu innego dnia wpisz datę istniejących folderów z `runs/`.

In [ ]:
from datetime import date
from IPython.display import Image, display
DAY = str(date.today())          # np. '2026-11-11' przy wznawianiu
RUNS = {'plain': f'{DAY}_unet_plain_long', 'ae': f'{DAY}_ae', 'corr': f'{DAY}_corr'}
EPOCHS = {'plain': 200, 'ae': 60, 'corr': 80}
BS = 16

def train(mode, resume=False, lr=3e-4):
    run = RUNS[mode]
    res = f'--resume "{P.RUNS / run / "last.pt"}"' if resume else ''
    !python src/train_unet.py --data "{DATA}" --mode {mode} --epochs {EPOCHS[mode]} --bs {BS} --lr {lr} --workers {WORKERS} --out "{P.RUNS}" --name {run} {res}
    prev = sorted((P.RUNS / run).glob('preview_ep*.png'))
    if prev:
        print(prev[-1]); display(Image(str(prev[-1])))
print(RUNS)

## 3. Zwykły U-Net od zera, 200 epok (~21 s/epokę na L4, ok. 70 min)
Po rozłączeniu: SETUP, 1, 2, potem `train('plain', resume=True)`.

In [ ]:
train('plain')

## 4. Autoenkoder bez skipów, 60 epok
Podgląd: RS A | RS B | wynik | błąd ×4 | wzorzec.

In [ ]:
train('ae')

## 5. Enkoder + warstwa korelacji, 80 epok

In [ ]:
train('corr')

## 6. Porównanie na val
Wiersze z wcześniejszych notatników są brane z ich `results.json` (najlepsza epoka), jeśli są na Drive.
`psnr` = obszar widoczny w A (dla `ae`/`corr` cały obraz), `psnr_full` = cały obraz (wspólna miara dla wszystkich).

In [ ]:
import json, pandas as pd
rows = []
for label, name in [('RAFT + geometria + U-Net (05)', 'raft_unet_ft'), ('zwykły U-Net 30 ep. (06)', 'unet_plain')]:
    rj = P.find_run(name, 'results.json')
    if rj:
        rows.append({'model': label, 'run': rj.parent.name, **json.load(open(rj))['val_metrics']})
for mode, run in RUNS.items():
    rj = P.RUNS / run / 'results.json'
    if rj.exists():
        r = json.load(open(rj))
        rows.append({'model': mode, 'run': run, 'best_epoch': r['best_epoch'], **r['val_metrics']})
cols = ['run', 'best_epoch', 'psnr_full', 'psnr', 'ssim', 'psnr_full_in', 'psnr_in']
tab = pd.DataFrame(rows).set_index('model')
display(tab[[c for c in cols if c in tab]].round(3))

Krzywe uczenia (val PSNR po epokach): czy zwykły U-Net się wypłaszcza, czy nadal rośnie.

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8, 4))
for mode, run in RUNS.items():
    f = P.RUNS / run / 'log.csv'
    if f.exists():
        log = pd.read_csv(f)
        ax.plot(log['epoch'], log['val_psnr_full'], label=f'{mode} ({run})')
old = P.find_run('unet_plain', 'log.csv')
if old:
    log = pd.read_csv(old); ax.plot(log['epoch'], log['val_psnr_full'], '--', label=f'plain 30 ep. ({old.parent.name})')
ax.axhline(35.37, color='k', ls=':', label='RAFT + geometria + U-Net')
ax.set_xlabel('epoka'); ax.set_ylabel('val PSNR, cały obraz [dB]'); ax.grid(alpha=0.3); ax.legend()
plt.show()

Ewaluacja z wynikami dla każdej próbki (do podziału na siłę RS jak w 06, sekcja 4):

In [ ]:
for mode, run in RUNS.items():
    ck = P.RUNS / run / 'best.pt'
    if ck.exists():
        !python src/train_unet.py --data "{DATA}" --eval "{ck}" --eval-json "{P.RUNS / run / 'eval.json'}" --per-sample "{P.RUNS / run / 'val_per_sample.csv'}" --bs {BS} --workers {WORKERS} > /dev/null
        print(run, json.load(open(P.RUNS / run / 'eval.json'))['metrics']['ms_per_pair'], 'ms/parę')